# Student Feedback Analysis with RAG

This notebook analyzes student feedback using LangChain agents with retrieval-augmented generation.

In [1]:
# pip install -q langchain langchain-groq langchain-community sentence-transformers chromadb pandas openpyxl python-dotenv langchain-huggingface langchain-chroma

import os
import sys
import pandas as pd

# Ensure required packages are installed in this notebook kernel
required_packages = [
    "langchain",
    "langchain-groq",
    "langchain-community",
    "sentence-transformers",
    "chromadb",
    "pandas",
    "openpyxl",
    "python-dotenv",
    "langchain-huggingface",
    "langchain-chroma",
]

os.system(f"{sys.executable} -m pip install -q " + " ".join(required_packages))

from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate

# Load environment variables from .env
load_dotenv(override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    print("WARNING: GROQ_API_KEY not found in .env file. Please set it.")
else:
    print("Groq API Key found.")

huggingface_api_key = os.getenv("HUGGINGFACE_API_KEY")
if not huggingface_api_key:
    print("WARNING: HUGGINGFACE_API_KEY not found in .env file. Please set it.")
else:
    print("Huggingface API Key found.")

You should consider upgrading via the '/Users/madams1/Documents/FAU/boilerplate/.venv/bin/python -m pip install --upgrade pip' command.
/Users/madams1/Documents/FAU/boilerplate/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/madams1/Documents/FAU/boilerplate/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Groq API Key found.
Huggingface API Key found.


In [ ]:

file_path = "feedback.xlsx"

if os.path.exists(file_path):
    df = pd.read_excel(file_path)
    print("Data loaded. First 5 rows:")
    print(df.head())
else:
    print(f"File not found: {file_path}. Please make sure the file is in the same directory.")

Data loaded. First 5 rows:
   z_last_digits                                    student_writeup
0         436967  I've taken a few AI-related classes before. Th...
1         503497  I want to get a good overview of artificial in...
2         607951  As a front-end engineer with 8 years experienc...
3         454016  In completing this Artificial Intelligence cou...
4         569976  Through this course, I aim to develop a robust...


In [ ]:

documents = []


for index, row in df.iterrows():

    content = "\n".join([f"{col}: {val}" for col, val in row.items() if pd.notna(val)])
    
    doc = Document(page_content=content, metadata={"row": index, "source": file_path})
    documents.append(doc)

print(f"Processed {len(documents)} documents.")

Processed 134 documents.


In [ ]:

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="student_feedback",
    persist_directory="./chroma_db"
)

print("Vector Database created.")

Vector Database created.


In [ ]:


@tool
def get_all_feedback() -> str:
    """Get all student feedback to analyze topics and themes"""
    docs = vectorstore.get()['documents']
    if isinstance(docs, list) and len(docs) > 0:
        response_content = "\n\n---\n\n".join(docs[:25])  # Get first 25 docs
    else:
        # Fallback: search with broad query
        docs = vectorstore.similarity_search("learning goals interests", k=30)
        response_content = "\n\n".join([d.page_content for d in docs])
    return f"Total feedback entries: {len(docs)}\n\n{response_content}"

@tool  
def search_feedback(query: str) -> str:
    """Search for specific feedback on a topic"""
    docs = vectorstore.similarity_search(query, k=10)
    response_content = "\n\n".join([d.page_content for d in docs])
    return response_content

print("Tools created.")

Tools created.


In [ ]:


# Initialize the LLM
llm = ChatGroq(model_name="meta-llama/llama-4-scout-17b-16e-instruct", api_key=groq_api_key)

# Initialize the tools
tools = [get_all_feedback, search_feedback]


# Create prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that analyzes student feedback. You have access to all student responses and should use them to provide thorough, accurate analysis."),
    ("placeholder", "{agent_scratchpad}"),
    ("human", "{input}"),
])

# Create and setup agent
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


print("Agent created and ready.")

Agent created and ready.


## Query 1: Find Two Most Similar Answers

In [ ]:

input_1 = "Using the complete student feedback data, find two most similar answers provided by students and explain why they are THE MOST similar."

print("\n" + "="*80)
print("QUERY 1: Finding Most Similar Student Answers")
print("="*80)

result_1 = agent_executor.invoke({"input": input_1})
answer_1 = result_1['output']

print("\n" + "-"*80)
print("RESULT 1:")
print("-"*80)
print(answer_1)


QUERY 1: Finding Most Similar Student Answers


> Entering new AgentExecutor chain...

Invoking: `get_all_feedback` with `{}`


Total feedback entries: 1340

z_last_digits: 436967
student_writeup: I've taken a few AI-related classes before. They taught me how to build models and some of the math, but the majority of the syllabus for this course covers topics I am not familiar with, such as search and knowledge representation. I'm excited to learn more about AI besides just neural networks! Although not included in the syllabus, I'd also love to touch on how GenAI works.

---

z_last_digits: 503497
student_writeup: I want to get a good overview of artificial intelligence and understand how it is used in real world applications. I expect to learn the main AI concepts, basic methods, and how they solve practical problems. I also want to understand where AI works well, and its limitations.

---

z_last_digits: 607951
student_writeup: As a front-end engineer with 8 years experience, I'm lo

## Query 2: Summarize Main Topics and Top 5 Themes

In [ ]:

input_2 = "Using the complete student feedback data, provide: 1) A summary of the main topics and interests students have about this course, 2) Identify and list the top 5 most common themes/topics students mentioned, 3) For each topic, briefly explain what students want to learn"

print("\n" + "="*80)
print("QUERY 2: Summary of Main Topics and Top 5 Themes")
print("="*80)

result_2 = agent_executor.invoke({"input": input_2})
answer_2 = result_2['output']

print("\n" + "-"*80)
print("RESULT 2:")
print("-"*80)
print(answer_2)


QUERY 2: Summary of Main Topics and Top 5 Themes


> Entering new AgentExecutor chain...

Invoking: `get_all_feedback` with `{}`


Total feedback entries: 1340

z_last_digits: 436967
student_writeup: I've taken a few AI-related classes before. They taught me how to build models and some of the math, but the majority of the syllabus for this course covers topics I am not familiar with, such as search and knowledge representation. I'm excited to learn more about AI besides just neural networks! Although not included in the syllabus, I'd also love to touch on how GenAI works.

---

z_last_digits: 503497
student_writeup: I want to get a good overview of artificial intelligence and understand how it is used in real world applications. I expect to learn the main AI concepts, basic methods, and how they solve practical problems. I also want to understand where AI works well, and its limitations.

---

z_last_digits: 607951
student_writeup: As a front-end engineer with 8 years experience, I'm

## Summary

Both queries have been completed. The first query identified the most similar student answers, while the second query provided a comprehensive analysis of main topics and the top 5 themes students are interested in.